In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gómez Alarcón | FECHA: 2026
# FASE: Paso 1. Gráficos multi-sensor (para 4 meses)
# DESCRIPCIÓN:
#   Carga datos de 4 sensores interiores (pasillo, colgado, ventana,
#   iluminación) y exteriores (Tª, humedad, radiación solar). Genera
#   scatter plots por mes (dic-mar) correlacionando Tª, CO2, humedad y
#   consumo, incluyendo relaciones interior-exterior.
#
# UBICACIÓN DE LOS SENSORES EN EL AULA 0.13:
#   - Corridor (Pasillo): Junto a la pared, orientado al pasillo.
#   - Hanging (Colgado): Colgando del techo en el centro del aula.
#   - Window (Ventana): Junto a la ventana que da al patio inteiror.
#   - Lighting (Iluminación): Integrado en el panel eléctrico del aula.
# ===================================================================

import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import numpy as np

# Directorio
OUTPUT_DIR = 'plots_scatters_4sensors'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# CARGAR TODOS LOS DATOS
# ==========================================
file_pattern = '013_data_*_week_*.xlsx'
all_files = sorted(glob.glob(file_pattern))

if not all_files:
    print(f"\nERROR: No files found matching '{file_pattern}'")
    exit()

print(f"\nFound {len(all_files)} weekly files")

print("\n" + "="*60)
print("LOADING ALL DATA...")
print("="*60)

all_records = []

for file_path in all_files:
    fname = os.path.basename(file_path)
    print(f"   {fname}...", end=" ")

    df = pd.read_excel(file_path)

    metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']
    date_cols = [c for c in df.columns if c not in metadata_cols]

    def get_series(df, variable, location):
        mask = (df['Variable_Measure'] == variable) & (df['Sensor_Location'] == location)
        row = df[mask]
        if len(row) > 0:
            return row[date_cols].iloc[0].astype(float)
        return pd.Series(np.nan, index=date_cols)

    # Corridor
    temp_corridor = get_series(df, 'Temperature', 'Corridor')
    co2_corridor  = get_series(df, 'CO2', 'Corridor')
    hum_corridor  = get_series(df, 'Humidity', 'Corridor')
    cons_corridor = get_series(df, 'Consumption', 'Corridor')  

    # Hanging
    temp_hanging = get_series(df, 'Temperature', 'Hanging')
    co2_hanging  = get_series(df, 'CO2', 'Hanging')
    hum_hanging  = get_series(df, 'Humidity', 'Hanging')
    cons_hanging = get_series(df, 'Consumption', 'Hanging')    

    # Window
    temp_window = get_series(df, 'Temperature', 'Window')
    co2_window  = get_series(df, 'CO2', 'Window')
    hum_window  = get_series(df, 'Humidity', 'Window')
    cons_window = get_series(df, 'Consumption', 'Window')      

    # Lighting
    temp_lighting = get_series(df, 'Temperature', 'Lighting')
    co2_lighting  = get_series(df, 'CO2', 'Lighting')
    hum_lighting  = get_series(df, 'Humidity', 'Lighting')
    cons_lighting = get_series(df, 'Consumption', 'Lighting')  

    # Outdoor
    temp_outdoor  = get_series(df, 'Temperature_Outdoor', 'Outdoor')
    hum_outdoor   = get_series(df, 'Humidity_Outdoor', 'Outdoor')
    solar_outdoor = get_series(df, 'Solar_Radiation', 'Outdoor')

    df_mini = pd.DataFrame({
        'Temperature_Corridor': temp_corridor.values,
        'CO2_Corridor':         co2_corridor.values,
        'Humidity_Corridor':    hum_corridor.values,
        'Consumption_Corridor': cons_corridor.values,
        'Temperature_Hanging':  temp_hanging.values,
        'CO2_Hanging':          co2_hanging.values,
        'Humidity_Hanging':     hum_hanging.values,
        'Consumption_Hanging':  cons_hanging.values,
        'Temperature_Window':   temp_window.values,
        'CO2_Window':           co2_window.values,
        'Humidity_Window':      hum_window.values,
        'Consumption_Window':   cons_window.values,
        'Temperature_Lighting': temp_lighting.values,
        'CO2_Lighting':         co2_lighting.values,
        'Humidity_Lighting':    hum_lighting.values,
        'Consumption_Lighting': cons_lighting.values,
        'Temperature_Outdoor':  temp_outdoor.values,
        'Humidity_Outdoor':     hum_outdoor.values,
        'Solar_Radiation':      solar_outdoor.values,
    }, index=pd.to_datetime(date_cols, errors='coerce'))

    all_records.append(df_mini)
    print(f"OK ({len(df_mini)} records)")

print("\n   Concatenating all data...")
df_all = pd.concat(all_records, ignore_index=False)
df_all = df_all.sort_index()
df_all['Month'] = df_all.index.month_name()

print(f"   Total records: {len(df_all)}")
print(f"   Months: {df_all['Month'].unique().tolist()}")

# Verificar datos de consumo
print("\n   Verificando datos de consumo:")
for loc in ['Corridor', 'Hanging', 'Window', 'Lighting']:
    col = f'Consumption_{loc}'
    if col in df_all.columns:
        n_valid = df_all[col].notna().sum()
        if n_valid > 0:
            print(f"      ✓ {loc}: {n_valid} registros válidos")
        else:
            print(f"      ✗ {loc}: SIN DATOS")
    else:
        print(f"      ✗ {loc}: columna no encontrada")

# ==========================================
# CONFIGURACIÓN
# ==========================================
month_order = ['December', 'January', 'February', 'March']
unique_months = [m for m in month_order if m in df_all['Month'].unique()]

sensor_colors = {
    'Corridor': '#2196F3',  # Azul
    'Hanging':  '#4CAF50',  # Verde
    'Window':   '#FF9800',  # Naranja
    'Lighting': '#9C27B0',  # Morado
}

sensor_markers = {
    'Corridor': 'o',  # Círculo
    'Hanging':  's',  # Cuadrado
    'Window':   '^',  # Triángulo
    'Lighting': 'D',  # Diamante
}

# ==========================================
# FUNCIONES DE GRÁFICOS
# ==========================================
def create_scatter_4sensors(var_name, x_vars_dict, y_vars_dict, x_label, y_label, suptitle, filename):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=False, sharey=False)
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.01)

    axes_flat = axes.flatten()
    month_positions = {'December': 0, 'January': 1, 'February': 2, 'March': 3}

    any_data = False

    for month, pos in month_positions.items():
        ax = axes_flat[pos]

        if month not in unique_months:
            ax.set_visible(False)
            continue

        data_month = df_all[df_all['Month'] == month]
        month_has_data = False

        for location in ['Corridor', 'Hanging', 'Window', 'Lighting']:
            x_col = x_vars_dict[location]
            y_col = y_vars_dict[location]

            if x_col not in data_month.columns or y_col not in data_month.columns:
                continue

            df_loc = data_month.dropna(subset=[x_col, y_col])

            if len(df_loc) < 3:
                continue

            month_has_data = True
            any_data = True

            ax.scatter(df_loc[x_col], df_loc[y_col],
                       c=sensor_colors[location],
                       marker=sensor_markers[location],
                       alpha=0.4, s=12, edgecolors='none',
                       label=location)

        if not month_has_data:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        ax.set_title(f'{month}  |  n={len(data_month)}', fontsize=11, fontweight='bold')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

    if not any_data:
        print(f"   SKIPPED: No data for {suptitle}")
        plt.close()
        return

    plt.tight_layout()
    file_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(file_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")

def create_scatter_outdoor_4sensors(x_var, y_vars_dict, x_label, y_label, suptitle, filename):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=False, sharey=False)
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.01)

    axes_flat = axes.flatten()
    month_positions = {'December': 0, 'January': 1, 'February': 2, 'March': 3}

    any_data = False

    for month, pos in month_positions.items():
        ax = axes_flat[pos]

        if month not in unique_months:
            ax.set_visible(False)
            continue

        data_month = df_all[df_all['Month'] == month]
        month_has_data = False

        for location in ['Corridor', 'Hanging', 'Window', 'Lighting']:
            y_col = y_vars_dict[location]

            if x_var not in data_month.columns or y_col not in data_month.columns:
                continue

            df_loc = data_month.dropna(subset=[x_var, y_col])

            if len(df_loc) < 3:
                continue

            month_has_data = True
            any_data = True

            ax.scatter(df_loc[x_var], df_loc[y_col],
                       c=sensor_colors[location],
                       marker=sensor_markers[location],
                       alpha=0.4, s=12, edgecolors='none',
                       label=location)

        if not month_has_data:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        ax.set_title(f'{month}  |  n={len(data_month)}', fontsize=11, fontweight='bold')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

    if not any_data:
        print(f"   SKIPPED: No data for {suptitle}")
        plt.close()
        return

    plt.tight_layout()
    file_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(file_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   SAVED: {filename}")

# ==========================================
# DICCIONARIOS DE VARIABLES
# ==========================================
temp_vars = {
    'Corridor': 'Temperature_Corridor', 
    'Hanging': 'Temperature_Hanging', 
    'Window': 'Temperature_Window',
    'Lighting': 'Temperature_Lighting'
}

co2_vars = {
    'Corridor': 'CO2_Corridor', 
    'Hanging': 'CO2_Hanging', 
    'Window': 'CO2_Window',
    'Lighting': 'CO2_Lighting'
}

hum_vars = {
    'Corridor': 'Humidity_Corridor', 
    'Hanging': 'Humidity_Hanging', 
    'Window': 'Humidity_Window',
    'Lighting': 'Humidity_Lighting'
}

cons_vars = {
    'Corridor': 'Consumption_Corridor', 
    'Hanging': 'Consumption_Hanging', 
    'Window': 'Consumption_Window',
    'Lighting': 'Consumption_Lighting'
}

# ==========================================
# GENERAR GRÁFICOS
# ==========================================
print("\n" + "="*60)
print("GENERATING SCATTER PLOTS (4 sensors per plot)")
print("="*60)

print("\n--- INTERIOR VARIABLES ---")

create_scatter_4sensors('CO2', temp_vars, co2_vars,
    'Temperature (°C)', 'CO₂ (ppm)',
    'CO₂ vs Temperature — Classroom 0.13 ',
    '01_CO2_vs_Temperature.png')

create_scatter_4sensors('Temperature', co2_vars, temp_vars,
    'CO₂ (ppm)', 'Temperature (°C)',
    'Temperature vs CO₂ — Classroom 0.13 ',
    '02_Temperature_vs_CO2.png')

create_scatter_4sensors('Humidity', co2_vars, hum_vars,
    'CO₂ (ppm)', 'Humidity (%)',
    'Humidity vs CO₂ — Classroom 0.13 ',
    '03_Humidity_vs_CO2.png')

create_scatter_4sensors('CO2', hum_vars, co2_vars,
    'Humidity (%)', 'CO₂ (ppm)',
    'CO₂ vs Humidity — Classroom 0.13 ',
    '04_CO2_vs_Humidity.png')

create_scatter_4sensors('Temperature', hum_vars, temp_vars,
    'Humidity (%)', 'Temperature (°C)',
    'Temperature vs Humidity — Classroom 0.13 ',
    '05_Temperature_vs_Humidity.png')

create_scatter_4sensors('Consumption', temp_vars, cons_vars,
    'Temperature (°C)', 'Consumption (kWh)',
    'Consumption vs Temperature — Classroom 0.13 ',
    '07_Consumption_vs_Temperature.png')

create_scatter_4sensors('Consumption', co2_vars, cons_vars,
    'CO₂ (ppm)', 'Consumption (kWh)',
    'Consumption vs CO₂ — Classroom 0.13 ',
    '08_Consumption_vs_CO2.png')

print("\n--- OUTDOOR VARIABLES ---")

create_scatter_outdoor_4sensors('Temperature_Outdoor', temp_vars,
    'Outdoor Temperature (°C)', 'Indoor Temperature (°C)',
    'Indoor vs Outdoor Temperature — Classroom 0.13 ',
    '11_Indoor_vs_Outdoor_Temperature.png')

create_scatter_outdoor_4sensors('Humidity_Outdoor', hum_vars,
    'Outdoor Humidity (%)', 'Indoor Humidity (%)',
    'Indoor vs Outdoor Humidity — Classroom 0.13 ',
    '12_Indoor_vs_Outdoor_Humidity.png')

create_scatter_outdoor_4sensors('Solar_Radiation', temp_vars,
    'Solar Radiation (W/m²)', 'Indoor Temperature (°C)',
    'Indoor Temperature vs Solar Radiation — Classroom 0.13 ',
    '13_Temperature_vs_Solar_Radiation.png')

create_scatter_outdoor_4sensors('Temperature_Outdoor', co2_vars,
    'Outdoor Temperature (°C)', 'Indoor CO₂ (ppm)',
    'Indoor CO₂ vs Outdoor Temperature — Classroom 0.13 ',
    '14_CO2_vs_Outdoor_Temperature.png')

create_scatter_outdoor_4sensors('Temperature_Outdoor', cons_vars,
    'Outdoor Temperature (°C)', 'Consumption (kWh)',
    'Consumption vs Outdoor Temperature — Classroom 0.13 ',
    '15_Consumption_vs_Outdoor_Temperature.png')

create_scatter_outdoor_4sensors('Solar_Radiation', cons_vars,
    'Solar Radiation (W/m²)', 'Consumption (kWh)',
    'Consumption vs Solar Radiation — Classroom 0.13 ',
    '16_Consumption_vs_Solar_Radiation.png')

print("\n" + "="*60)
print("4 SENSORS PLOTS COMPLETE")
print("="*60)


Found 13 weekly files

LOADING ALL DATA...
   013_data_December_2025_week_4_days_22-31.xlsx... OK (2563 records)
   013_data_February_2026_week_1_days_1-7.xlsx... OK (1750 records)
   013_data_February_2026_week_2_days_8-14.xlsx... OK (1838 records)
   013_data_February_2026_week_3_days_15-21.xlsx... OK (1710 records)
   013_data_February_2026_week_4_days_22-28.xlsx... OK (5390 records)
   013_data_January_2026_week_1_days_1-7.xlsx... OK (1580 records)
   013_data_January_2026_week_2_days_8-14.xlsx... OK (1474 records)
   013_data_January_2026_week_3_days_15-21.xlsx... OK (1375 records)
   013_data_January_2026_week_4_days_22-31.xlsx... OK (2616 records)
   013_data_March_2026_week_1_days_1-7.xlsx... OK (6656 records)
   013_data_March_2026_week_2_days_8-14.xlsx... OK (6659 records)
   013_data_March_2026_week_3_days_15-21.xlsx... OK (6607 records)
   013_data_March_2026_week_4_days_22-31.xlsx... OK (6973 records)

   Concatenating all data...
   Total records: 47191
   Months: ['Dece